[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/31_transformers/02_multi_head_and_positional_encoding.ipynb)

# Bài 02 — Multi-head attention, positional encoding và khối Transformer

**🎯 Mục tiêu**
- Cài **multi-head attention** bằng PyTorch, kiểm tra khớp `nn.MultiheadAttention`; đếm tham số.
- **Mã hoá vị trí**: sinusoidal (tính chất dịch chuyển), learned, **RoPE** (vị trí tương đối qua phép quay).
- Lắp **khối Transformer**: attention + FFN + residual + LayerNorm (pre-LN vs post-LN).
- Thí nghiệm: không có mã hoá vị trí, Transformer **không thể** đảo/sắp xếp/dịch chuỗi.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/31_transformers

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
print("thiết bị:", device)

## 1. Multi-head attention

Một đầu attention cho mỗi token **một** phân phối trọng số — một "kiểu quan hệ". Ngôn ngữ cần nhiều quan hệ cùng lúc (chủ ngữ–động từ, tính từ–danh từ, token liền trước...). **Multi-head**: chia $d_{model}$ thành $h$ đầu, mỗi đầu kích thước $d_k = d_{model}/h$, chạy attention song song với phép chiếu riêng, rồi ghép lại và chiếu ra:

$$\text{head}_i = \text{Attention}(XW_Q^{(i)}, XW_K^{(i)}, XW_V^{(i)}), \qquad \text{MHA}(X) = [\text{head}_1; \dots; \text{head}_h]\,W_O$$

Chi phí gần như bằng một đầu kích thước $d_{model}$; số tham số **không phụ thuộc** $h$: $4(d_{model}^2 + d_{model})$ (ba phép chiếu Q, K, V và $W_O$, có bias). Cài đặt: một phép chiếu chung $d \to 3d$, rồi `view` thành $(B, h, T, d_k)$.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)                     # W_Q, W_K, W_V gộp
        self.proj = nn.Linear(d_model, d_model)                        # W_O

    def forward(self, x, mask=None, return_weights=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=-1)
        q, k, v = (t.view(B, T, self.h, self.d_k).transpose(1, 2) for t in (q, k, v))   # (B, h, T, d_k)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)        # (B, h, T, T)
        if mask is not None:                                           # mask bool: True = được nhìn
            scores = scores.masked_fill(~mask, float("-inf"))
        w = torch.softmax(scores, dim=-1)
        out = (w @ v).transpose(1, 2).reshape(B, T, D)                 # ghép các đầu
        out = self.proj(out)
        return (out, w) if return_weights else out


torch.manual_seed(0)
mha = MultiHeadAttention(16, 4)
ref = nn.MultiheadAttention(16, 4, batch_first=True)
with torch.no_grad():                                                  # chép trọng số sang bản của PyTorch
    ref.in_proj_weight.copy_(mha.qkv.weight)
    ref.in_proj_bias.copy_(mha.qkv.bias)
    ref.out_proj.weight.copy_(mha.proj.weight)
    ref.out_proj.bias.copy_(mha.proj.bias)
x = torch.randn(2, 5, 16)
print(f"sai lệch so với nn.MultiheadAttention: {(mha(x) - ref(x, x, x)[0]).abs().max().item():.1e}")
for h in [1, 4, 8]:
    print(f"d_model = 16, {h} đầu: {sum(p.numel() for p in MultiHeadAttention(16, h).parameters())} tham số (= 4·(16² + 16) = {4 * (16**2 + 16)})")

## 2. Mã hoá vị trí

Bài 01: self-attention coi chuỗi là tập hợp. Cách đơn giản nhất để đưa thứ tự vào: **cộng** một vector phụ thuộc vị trí vào embedding của token: $\mathbf x_i = \text{emb}(t_i) + \text{PE}(i)$.

### 2.1. Sinusoidal (Vaswani et al., 2017)

$$\text{PE}(p, 2i) = \sin\!\left(\frac{p}{10000^{2i/d}}\right), \qquad \text{PE}(p, 2i+1) = \cos\!\left(\frac{p}{10000^{2i/d}}\right)$$

Mỗi cặp chiều là một "kim đồng hồ" quay với tần số riêng — từ rất nhanh (chiều đầu) đến rất chậm (chiều cuối), giống các chữ số của một số đếm. Không có tham số, tính được cho mọi vị trí.

In [ ]:
def sinusoidal_encoding(T, d):
    pos = torch.arange(T)[:, None]
    freq = 1.0 / 10000 ** (torch.arange(0, d, 2) / d)
    pe = torch.zeros(T, d)
    pe[:, 0::2] = torch.sin(pos * freq)
    pe[:, 1::2] = torch.cos(pos * freq)
    return pe


pe = sinusoidal_encoding(100, 64)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
im = axes[0].imshow(pe.T, aspect="auto", cmap="RdBu")
axes[0].set_xlabel("vị trí p")
axes[0].set_ylabel("chiều")
axes[0].set_title("PE sinusoidal (100 vị trí × 64 chiều)")
plt.colorbar(im, ax=axes[0])
sim = pe @ pe.T
axes[1].plot(sim[50].numpy())
axes[1].axvline(50, color="gray", ls=":")
axes[1].set_xlabel("vị trí q")
axes[1].set_title("PE(50) · PE(q): lớn nhất tại q = 50, giảm dần theo khoảng cách")
plt.tight_layout()
plt.show()

Hai tính chất:
- **Dịch chuyển tuyến tính**: với mỗi cặp chiều tần số $\omega$, $\text{PE}(p + k)$ là $\text{PE}(p)$ **quay** một góc $\omega k$ — một phép biến đổi tuyến tính chỉ phụ thuộc $k$. Mô hình có thể học "nhìn token cách $k$ vị trí" bằng phép chiếu tuyến tính.
- $\text{PE}(p)\cdot\text{PE}(q) = \sum_i\cos(\omega_i(p - q))$ chỉ phụ thuộc **khoảng cách** $p - q$.

In [ ]:
k = 7
freq = 1.0 / 10000 ** (torch.arange(0, 64, 2) / 64)
R = torch.zeros(64, 64)                                                # ma trận quay khối, chỉ phụ thuộc k
for i, w in enumerate(freq):
    c, s = math.cos(w * k), math.sin(w * k)
    R[2 * i:2 * i + 2, 2 * i:2 * i + 2] = torch.tensor([[c, -s], [s, c]])   # [sin, cos] · R = [sin(ω(p+k)), cos(ω(p+k))]
print(f"max |PE(p + 7) − PE(p)·R_7| với mọi p: {(pe[k:] - pe[:-k] @ R).abs().max().item():.1e}")
print(f"PE(10)·PE(17) = {pe[10] @ pe[17]:.4f} | PE(60)·PE(67) = {pe[60] @ pe[67]:.4f}  (cùng khoảng cách 7)")

### 2.2. Learned positional embedding

Một bảng `nn.Embedding(max_len, d)` học như mọi tham số khác (GPT-2, BERT, ViT). Đơn giản, thường tốt ngang sinusoidal, nhưng **không có** vector cho vị trí vượt `max_len` khi train.

### 2.3. RoPE — Rotary Position Embedding

Thay vì cộng vào embedding, **quay** query và key theo vị trí trước khi tính tích vô hướng (Su et al., 2021; dùng trong LLaMA, Qwen, Mistral...). Với mỗi cặp chiều $(x_{2i}, x_{2i+1})$ của $\mathbf q$ ở vị trí $m$, quay góc $m\theta_i$. Vì phép quay bảo toàn tích vô hướng tương đối: $\langle R_m\mathbf q, R_n\mathbf k\rangle = \langle\mathbf q, R_{n-m}\mathbf k\rangle$ — điểm attention chỉ phụ thuộc **vị trí tương đối** $n - m$.

In [ ]:
def apply_rope(x, positions, base=10000.0):
    """x: (..., T, d) với d chẵn; quay từng cặp chiều (2i, 2i+1) một góc pos·θ_i."""
    d = x.shape[-1]
    theta = 1.0 / base ** (torch.arange(0, d, 2, dtype=x.dtype) / d)
    ang = positions[:, None].to(x.dtype) * theta                      # (T, d/2)
    cos, sin = torch.cos(ang), torch.sin(ang)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * cos - x2 * sin
    out[..., 1::2] = x1 * sin + x2 * cos
    return out


torch.manual_seed(0)
q, kv = torch.randn(1, 16, dtype=torch.float64), torch.randn(1, 16, dtype=torch.float64)
for m, n in [(3, 8), (40, 45), (100, 105)]:
    score = apply_rope(q, torch.tensor([m])) @ apply_rope(kv, torch.tensor([n])).T
    print(f"q ở vị trí {m:3d}, k ở vị trí {n:3d} (cách 5): điểm = {score.item():.6f}")
print(f"chuẩn trước/sau khi quay: {q.norm():.6f} / {apply_rope(q, torch.tensor([37])).norm():.6f}")

Ba cặp vị trí cùng khoảng cách 5 cho **cùng một điểm** — RoPE đưa vị trí tương đối trực tiếp vào attention, không thay đổi chuẩn vector. Các biến thể khác: **ALiBi** (cộng một độ lệch tuyến tính $-m\cdot|i - j|$ vào điểm), **T5 relative bias** (bias học được theo khoảng cách).

## 3. Khối Transformer

Mỗi khối gồm hai lớp con, mỗi lớp con bọc bởi **residual** và **LayerNorm**:
1. **Multi-head self-attention** — trao đổi thông tin **giữa** các token.
2. **Feed-forward** (FFN, MLP 2 lớp, thường $d_{ff} = 4d$, GELU) — xử lý **từng** token độc lập; chiếm khoảng 2/3 tham số.

**Pre-LN** (GPT-2 trở đi): $\mathbf x \leftarrow \mathbf x + \text{MHA}(\text{LN}(\mathbf x))$, $\mathbf x \leftarrow \mathbf x + \text{FFN}(\text{LN}(\mathbf x))$ — đường residual "sạch" từ đầu đến cuối, train ổn định với mạng sâu. **Post-LN** (bài báo gốc): $\mathbf x \leftarrow \text{LN}(\mathbf x + \text{MHA}(\mathbf x))$ — thường cần **warmup** learning rate để không phân kỳ (Xiong et al., 2020).

Góc nhìn **residual stream**: mỗi token có một "dòng" vector chạy qua các khối; attention và FFN **đọc** từ dòng và **cộng** thông tin vào — giống ResNet (chương 29).

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_mult=4):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, ff_mult * d_model), nn.GELU(), nn.Linear(ff_mult * d_model, d_model))

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)                           # pre-LN
        return x + self.ffn(self.ln2(x))


block = TransformerBlock(64, 4)
n_attn = sum(p.numel() for p in block.attn.parameters())
n_ffn = sum(p.numel() for p in block.ffn.parameters())
n_all = sum(p.numel() for p in block.parameters())
print(f"khối d_model = 64: {n_all:,} tham số | attention {n_attn:,} ({n_attn / n_all:.0%}) | FFN {n_ffn:,} ({n_ffn / n_all:.0%})")
print("đầu vào/ra cùng shape:", tuple(block(torch.randn(2, 10, 64)).shape))

## 4. Thí nghiệm: vị trí có quan trọng không?

Encoder Transformer 2 khối ($d = 64$, 4 đầu) đọc 12 chữ số và xuất 12 chữ số (mỗi vị trí một lớp 0–9). Ba bài toán đều cần biết **vị trí**:
- **reverse**: đảo ngược chuỗi; **sort**: sắp xếp tăng dần; **shift**: dịch vòng phải 1 vị trí.

Ba biến thể: không mã hoá vị trí, sinusoidal, learned. 2 000 bước Adam, batch 128.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, pos_kind, vocab=10, T=12, d=64, n_heads=4, n_layers=2):
        super().__init__()
        self.pos_kind = pos_kind
        self.emb = nn.Embedding(vocab, d)
        if pos_kind == "learned":
            self.pos = nn.Parameter(torch.randn(T, d) * 0.02)
        elif pos_kind == "sinusoidal":
            self.register_buffer("pos", sinusoidal_encoding(T, d))
        self.blocks = nn.ModuleList([TransformerBlock(d, n_heads) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)

    def forward(self, x):
        h = self.emb(x)
        if self.pos_kind != "none":
            h = h + self.pos[: x.shape[1]]
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln(h))


TASKS = {"reverse": lambda x: x.flip(1), "sort": lambda x: x.sort(1).values, "shift": lambda x: torch.roll(x, 1, 1)}


def run_task(task, pos_kind, T=12, steps=2000, seed=0):
    torch.manual_seed(seed)
    model = Encoder(pos_kind, T=T).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    f = TASKS[task]
    for _ in range(steps):
        x = torch.randint(0, 10, (128, T), generator=g)
        opt.zero_grad()
        nn.functional.cross_entropy(model(x.to(device)).reshape(-1, 10), f(x).to(device).reshape(-1)).backward()
        opt.step()
    x = torch.randint(0, 10, (2000, T), generator=torch.Generator().manual_seed(123))
    with torch.no_grad():
        correct = model(x.to(device)).argmax(-1).cpu() == f(x)
    return correct.float().mean().item(), correct.all(1).float().mean().item()


results = {}
for task in TASKS:
    for pos_kind in ["none", "sinusoidal", "learned"]:
        t0 = time.perf_counter()
        results[task, pos_kind] = run_task(task, pos_kind)
        tok, exact = results[task, pos_kind]
        print(f"{task:8s} | {pos_kind:10s}: accuracy theo token = {tok:.3f} | khớp cả chuỗi = {exact:.3f} | {time.perf_counter() - t0:.0f}s")

- **Không mã hoá vị trí**: cả ba bài đều thất bại (khoảng 26% theo token, 0% cả chuỗi). Không phải do thiếu dung lượng: mô hình hoán vị-đẳng biến, nên hai vị trí có **cùng chữ số** luôn nhận **cùng dự đoán** — điều tốt nhất nó làm được là đoán chữ số xuất hiện nhiều trong chuỗi.
- **Có mã hoá vị trí**: cả ba bài đạt (gần) 100%. Sort khó nhất — cần cả so sánh nội dung lẫn vị trí đầu ra; khi soạn bài, có lần chạy (khác biệt nhỏ về số học) mà bản sinusoidal mới đạt ~70% khớp cả chuỗi sau 2 000 bước: trên bài khó, kết quả nhạy với khởi tạo và ngân sách huấn luyện — nên so sánh trên nhiều seed trước khi kết luận cách mã hoá nào tốt hơn.

Lưu ý ngược lại: với dữ liệu **là tập hợp** (bỏ qua thứ tự là đúng), không cần — và không nên — thêm mã hoá vị trí.

## 🧠 Kiểm tra nhanh

1. Multi-head attention khác single-head thế nào? Số tham số có tăng theo số đầu không?
2. Viết công thức PE sinusoidal. Tính chất nào giúp mô hình học vị trí tương đối?
3. RoPE đưa vị trí vào attention bằng cách nào? Vì sao điểm attention chỉ phụ thuộc khoảng cách?
4. Một khối Transformer gồm những gì? Pre-LN khác post-LN thế nào?
5. Vì sao Transformer không có mã hoá vị trí không thể đảo ngược chuỗi?

## 📌 Tóm tắt

- Multi-head: $h$ attention song song trên các không gian con $d/h$ chiều, ghép rồi chiếu; tham số $4(d^2 + d)$ bất kể $h$.
- Mã hoá vị trí: sinusoidal (không tham số, dịch chuyển = phép quay), learned (bảng học, giới hạn `max_len`), RoPE/ALiBi (vị trí tương đối, phổ biến trong LLM hiện đại).
- Khối Transformer = MHA + FFN, mỗi phần có residual + LayerNorm; pre-LN ổn định hơn.
- Không có vị trí, Transformer chỉ thấy tập hợp — thất bại hoàn toàn trên reverse/sort/shift.

## 📚 Tài liệu tham khảo

- Vaswani et al. (2017) — *Attention Is All You Need*
- Su et al. (2021) — *RoFormer: Enhanced Transformer with Rotary Position Embedding*
- Press, Smith & Lewis (2022) — *Train Short, Test Long: Attention with Linear Biases (ALiBi)*
- Xiong et al. (2020) — *On Layer Normalization in the Transformer Architecture*
- Elhage et al. (2021) — *A Mathematical Framework for Transformer Circuits* (Anthropic)